---
title: Automatic Differentiation
short_title: Automatic Differentiation
subject: DEEP LEARNING
---

## Introduction

Recall that all operations must be defined with specific local gradient computation for BP to work. In this section, we will implement a minimal **autograd engine** for creating computational graphs. This starts with the base `Node` class which has a `data` attribute for storing output and a `grad` attribute for storing the global gradient. Furthermore, the base class defines a `backward` method to solve for `grad` as described above.

In [1]:
%matplotlib inline
%config InlineBackend.figure_format = "svg"
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

import math
import random
import numpy as np
import pandas as pd

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

def get_device(): 
    return (
        torch.device("cuda:0") if torch.cuda.is_available() else (
            torch.device("mps") if torch.mps.is_available() else 
                torch.device("cpu")
        )
    )

DEVICE = get_device()
print(f"Using device: {DEVICE}")

In [2]:
def savefig(filename: str):
    plt.savefig(f"./plots/{filename}.svg", bbox_inches="tight")
    plt.close("all");

def directive(filename, caption_topic="", caption=""):
    if caption_topic:
        fig_caption = f"**{caption_topic}.** {caption}"
    else:
        fig_caption = caption

    print(f"""
:::{{figure}} ./plots/{filename}.svg
---
name: {filename}
width: 100%
align: center
---
{fig_caption}
:::
    """)

# remove this! having caption args useful in copilot

### Compute node class

In [3]:
from typing import final
from collections import OrderedDict


class Node:
    def __init__(self, data, parents=()):
        self.data = data
        self.grad = 0               # ∂loss / ∂self
        self._parents = parents     # parent -> self

    @final
    def sorted_nodes(self):
        """Return topologically sorted nodes with self as root."""
        topo = OrderedDict()

        def dfs(node):
            if node not in topo:
                for parent in node._parents:
                    dfs(parent)

                topo[node] = None

        dfs(self)
        return reversed(topo)


    @final
    def backward(self):
        """Send global grads backward to parent nodes."""
        self.grad = 1.0
        for node in self.sorted_nodes():
            for parent in node._parents:
                parent.grad += node.grad * node._local_grad(parent)


    def _local_grad(self, parent) -> float:
        """Calculate local grads ∂self / ∂parent."""
        raise NotImplementedError("Base node has no parents.")


    def __add__(self, node):
        return BinaryOpNode(self, node, op="+")

    def __mul__(self, node):
        return BinaryOpNode(self, node, op="*")

    def __pow__(self, n):
        assert isinstance(n, (int, float)) and n != 1
        return PowOp(self, n)

    def relu(self):
        return ReLUNode(self)

    def tanh(self):
        return TanhNode(self)

    def __neg__(self):
        return self * Node(-1)

    def __sub__(self, node):
        return self + (-node)

### Supported ops

Next, we define the **supported operations**. Only a handful are needed to implement a FNN:

In [4]:
class BinaryOpNode(Node):
    def __init__(self, x, y, op: str):
        """Binary operation between two nodes."""
        ops = {"+": lambda x, y: x + y, "*": lambda x, y: x * y}
        self._op = op
        super().__init__(ops[op](x.data, y.data), (x, y))

    def _local_grad(self, parent):
        if self._op == "+":
            return 1.0

        elif self._op == "*":
            i = self._parents.index(parent)
            coparent = self._parents[1 - i]
            return coparent.data

    def __repr__(self):
        return self._op


class ReLUNode(Node):
    def __init__(self, x):
        data = x.data * int(x.data > 0.0)
        super().__init__(data, (x,))

    def _local_grad(self, parent):
        return float(parent.data > 0)

    def __repr__(self):
        return "relu"


class TanhNode(Node):
    def __init__(self, x):
        data = math.tanh(x.data)
        super().__init__(data, (x,))

    def _local_grad(self, parent):
        return 1 - self.data**2

    def __repr__(self):
        return "tanh"


class PowOp(Node):
    def __init__(self, x, n):
        self.n = n
        data = x.data**self.n
        super().__init__(data, (x,))

    def _local_grad(self, parent):
        return self.n * parent.data ** (self.n - 1)

    def __repr__(self):
        return f"** {self.n}"

**NOTE:** Circular definition is weird but fine since references are resolved at runtime.

### Graph vizualization

The next two functions help to visualize networks. The `trace` function just walks backward into the graph to collect all nodes and edges. This is used by the `draw_graph` which first draws all nodes, then draws all edges. For compute nodes we add a small juncture node which contains the name of the operation.

In [5]:
from graphviz import Digraph


def trace(root):
    """Builds a set of all nodes and edges in a graph."""
    # https://github.com/karpathy/micrograd/blob/master/trace_graph.ipynb

    nodes = set()
    edges = set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for parent in v._parents:
                edges.add((parent, v))
                build(parent)

    build(root)
    return nodes, edges


def draw_graph(root, savename=None):
    """Build diagram of computational graph."""

    dot = Digraph(format="svg", graph_attr={"rankdir": "LR"})  # LR = left to right
    nodes, edges = trace(root)
    for n in nodes:
        # Add node to graph
        uid = str(id(n))
        dot.node(name=uid, label=f"data={n.data:.3f} | grad={n.grad:.4f}", shape="record")

        # Connect node to op node if operation
        # e.g. if (5) = (2) + (3), then draw (5) as (+) -> (5).
        if len(n._parents) > 0:
            dot.node(name=uid + str(n), label=str(n))
            dot.edge(uid + str(n), uid)

    for child, v in edges:
        # Connect child to the op node of v
        dot.edge(str(id(child)), str(id(v)) + str(v))

    if savename:
        dot.render(filename=savename, directory="./plots", format="svg", cleanup=True)
    else:
        return dot

Creating graph for a dense unit. Observe that `x1` has a degree of 2 since it has two children.

In [6]:
w0 = Node(-1.0)
w1 = Node(2.0)
b  = Node(4.0)
x  = Node(2.0)
t  = Node(3.0)

z = w0 * x + w1 * x + b
u = z.tanh()
y = z.relu()

draw_graph(y)

Gradients all check out:

In [7]:
y.backward()
draw_graph(y)

Note that `u` is not shown in the graph and `u.grad` is zero since `y` has no dependence on `u`:

In [8]:
u.grad

Moreover, gradients on shared parameters **accumulate** with multiple inputs:

In [9]:
x1 = Node(1.7)
z1 = w0 * x1 + w1 * x1 + b
y1 = z1.relu()
y1.backward()
draw_graph(y1)

## Neural network library

Here we construct the neural network module. The `Module` class defines an abstract class that maintains a list of the parameters used in forward pass implemented in `__call__`. The decorator `@final` is to prevent any inheriting class from overriding the methods as doing so would result in a warning (or an error with a type checker).

In [10]:
from abc import ABC, abstractmethod

class Module(ABC):
    def __init__(self):
        self._parameters = []

    @final
    def parameters(self) -> list:
        return self._parameters

    @abstractmethod
    def __call__(self, x: list):
        pass

    @final
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0

The `_parameters` attribute is defined so that the parameter list is not constructed at each call of the `parameters()` method. Implementing layers from neurons:

In [11]:
class Neuron(Module):
    def __init__(self, n_in, activation=None):
        self.n_in = n_in
        self.act = activation

        self.w = [Node(random.random()) for _ in range(n_in)]
        self.b = Node(0.0)
        self._parameters = self.w + [self.b]

    def __call__(self, x: list):
        assert len(x) == self.n_in
        out = sum((x[j] * self.w[j] for j in range(self.n_in)), start=self.b)
        if self.act is not None:
            if self.act == "tanh":
                out = out.tanh()
            elif self.act == "relu":
                out = out.relu()
            else:
                raise NotImplementedError("Activation not supported.")
        return out

    def __repr__(self):
        return f"{self.act if self.act is not None else 'linear'}({len(self.w)})"


class Layer(Module):
    def __init__(self, n_in, n_out, *args):
        self.neurons = [Neuron(n_in, *args) for _ in range(n_out)]
        self._parameters = [p for n in self.neurons for p in n.parameters()]

    def __call__(self, x: list):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def __repr__(self):
        return f"Layer[{', '.join(str(n) for n in self.neurons)}]"


class MLP(Module):
    def __init__(self, n_in, n_outs, activation=None):
        sizes = [n_in] + n_outs
        self.layers = []
        for i in range(len(n_outs)):
            act = activation if i < len(n_outs) - 1 else None
            layer = Layer(sizes[i], sizes[i + 1], act)
            self.layers.append(layer)

        self._parameters = [p for layer in self.layers for p in layer.parameters()]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def __repr__(self):
        return f"MLP[{', '.join(str(layer) for layer in self.layers)}]"

Testing model init and model call. Note that the final node has no activation:

In [12]:
model = MLP(n_in=1, n_outs=[2, 4, 1], activation="relu")
x = Node(1.0)
pred = model([x])
pred.backward()

print(model)
print(pred.data)

In [13]:
draw_graph(pred, savename="02-mlp_graph")

:::{figure} ./plots/02-mlp_graph.svg
---
name: 02-mlp_graph
width: 100%
align: center
---
**Graph of a MLP with ReLU activations.** The output is a single node.
:::

## Training from scratch

Our task is to learn the ff. dataset consisting of noisy measurements around a quadratic curve:

In [14]:
N = 1500
X = np.linspace(-1, 5, N)
Y = X ** 2 + 0.5 * np.random.normal(size=N, scale=3)

Helper for loading the samples:

In [15]:
class DataLoader:
    def __init__(self, dataset, batch_size):
        """Iterate over a partition of the dataset."""
        self.batch_size = batch_size
        self.dataset = [(Node(x), Node(y)) for x, y in dataset]
    
    def load(self):
        return random.sample(self.dataset, self.batch_size)

    def __len__(self):
        return len(self.dataset)

The function `optim_step` implements one step of SGD. Note that gradients accumulate, so we can implement batch size. But for simplicity, we only consider one training instance per update. Here `loss_fn` is the MSE between two nodes.

In [16]:
def optim_step(model, lr=1.0):
    for p in model.parameters():
        p.data -= lr * p.grad 

def loss_fn(y_pred, y_true):
    return (y_pred - y_true) ** 2

Running the training algorithm:

In [17]:
def train(model, dataset, steps, lr=0.1, batch_size=32):
    dataloader = DataLoader(dataset, batch_size)
    history = []
    for _ in tqdm(range(steps)):
        for x, y in dataloader.load():
            loss = loss_fn(model([x]), y)
            loss.backward()
            optim_step(model, lr=lr)
            
            model.zero_grad()
            history.append(loss.data)
    
    return history

dataset = list(zip(X, Y))
model = MLP(1, [8, 4, 1], "tanh")
losses = train(model, dataset, lr=0.003, batch_size=8, steps=5000)

Loss curve moving average decreasing:

In [18]:
w = 50
loss_avg = np.array(losses).reshape(-1, w).mean(axis=1)

In [19]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(loss_avg, label="train")
ax.grid(linestyle="dotted", alpha=0.6)
ax.set_ylabel("MSE")
ax.set_xlabel("steps")
ax.legend()
savefig("02-loss");

:::{figure} ./plots/02-loss.svg
---
name: 02-loss
width: 100%
align: center
---
**Decreasing loss indicates learning.** Mean squared error (MSE) during training.
:::


Model learned: ヾ( ˃ᴗ˂ )◞ • *✰

In [20]:
preds = [model([Node(x)]).data for x in X]

In [21]:
plt.figure(figsize=(6, 4))
plt.scatter(X, Y, label="data", s=2, alpha=0.7, color="black")
plt.plot(X, preds, label="model", color="C1", linewidth=2)

plt.ylabel("y")
plt.xlabel("x")
plt.grid(linestyle="dotted")
plt.legend()
savefig("02-predictions");

:::{figure} ./plots/02-predictions.svg
---
name: 02-predictions
width: 75%
align: center
---
**Predictions appear accurate.** Model predictions on the training data.
:::


In [22]:
x = Node(1.0)
pred = model([x])

draw_graph(pred, savename="02-trained_model_graph")

:::{figure} ./plots/02-trained_model_graph.svg
---
name: 02-trained_model_graph
width: 100%
align: center
---
**Visualizing the trained model.** Model is huge under the hood!
:::

## Appendix: Benchmarking

Recall BP has time and memory complexity that is **linear** in network size. This assumes each node executes in constant time and the outputs are stored. Moreover, the gradient should never be asymptotically slower than the function (assuming local gradient computation takes constant time). Testing this here empirically.


In [23]:
import time

x = [Node(1.0)] * 3
network_size  = []
fwd_times = {}
bwd_times = {}

for i in tqdm(range(10)):
    nouts = [200] * (i + 1) + [1]
    model = MLP(n_in=3, n_outs=nouts, activation="relu")

    fwd_times[i] = []
    bwd_times[i] = []
    for j in range(5):
        t0 = time.process_time()
        pred = model(x)
        t1 = time.process_time()
        fwd_times[i].append(t1 - t0)

        t0 = time.process_time()
        pred.grad = 1.0
        pred.backward()
        t1 = time.process_time()
        bwd_times[i].append(t1 - t0)

    network_size.append(len(model.parameters()) + sum(nouts) + 3)

In [24]:
for i, size in enumerate(network_size):
    for j in range(5):
        # add label once
        if j == 0 and i == 0:
            plt.scatter(size, fwd_times[i][j], color="C0", edgecolor="black", s=15, label="forward")
            plt.scatter(size, bwd_times[i][j], color="C1", edgecolor="black", s=15, label="backward")
        else:
            plt.scatter(size, fwd_times[i][j], color="C0", edgecolor="black", s=15)
            plt.scatter(size, bwd_times[i][j], color="C1", edgecolor="black", s=15)

plt.legend()
plt.xlabel("num hidden")
plt.ylabel("time (s)")
plt.ticklabel_format(axis="x", style="sci", scilimits=(3, 3))
plt.grid(linestyle="dotted")
savefig("02-benchmark");

:::{figure} ./plots/02-benchmark.svg
---
name: 02-benchmark
width: 100%
align: center
---
**Time vs. network size.** Time complexity for forward and backward pass are both roughly linear in network size.
:::

## Appendix: Testing with `autograd`

The `autograd` package allows automatic differentiation by building computational graphs on the fly every time we pass data through our model. Autograd tracks which data combined through which operations to produce the output. This allows us to take derivatives over ordinary imperative code. This functionality is consistent with the memory and time requirements outlined above for BP.

**Scalars.** Here we calculate $\mathsf{y} = \boldsymbol{\mathsf x}^\top \boldsymbol{\mathsf x} = \sum_i {\boldsymbol{\mathsf{x}}_i}^2$ where the initialized tensor $\boldsymbol{\mathsf{x}}$ initially has no gradient (i.e. `None`). Calling backward on $\mathsf{y}$ results in gradients being stored on the leaf tensor $\boldsymbol{\mathsf{x}}.$ Note that unlike our implementation, there is no need to set `y.grad = 1.0`. Moreover, doing so would result in an error as $\mathsf{y}$ is not a [leaf node](https://pytorch.org/docs/stable/generated/torch.Tensor.is_leaf.html) in the graph.

In [25]:
x = torch.arange(4, dtype=torch.float, requires_grad=True)
print(x.grad)

y = x.reshape(1, -1) @ x 
y.backward() 
print((x.grad == 2*x).all().item())

**Vectors.** Let $\boldsymbol{\mathsf y} = g(\boldsymbol{\mathsf x})$ and let $\boldsymbol{{\mathsf v}}$ be a vector having the same length as $\boldsymbol{\mathsf y}.$ Then `y.backward(v)` calculates
$\sum_i {\boldsymbol{\mathsf v}}_i \frac{\partial {\boldsymbol{\mathsf y}}_i}{\partial {\boldsymbol{\mathsf x}}_j}$
resulting in a vector of same length as $\boldsymbol{\mathsf{x}}$ stored in `x.grad`. Note that the terms on the right are the local gradients. Setting ${\boldsymbol{\mathsf v}} = \frac{\partial \mathcal{L} }{\partial \boldsymbol{\mathsf y}}$ gives us the vector $\frac{\partial \mathcal{L} }{\partial \boldsymbol{\mathsf x}}.$ Below $\boldsymbol{\mathsf y}(\boldsymbol{\mathsf x}) = [x_0, x_1].$

In [26]:
x = torch.rand(size=(4,), dtype=torch.float, requires_grad=True)
v = torch.rand(size=(2,), dtype=torch.float)
y = x[:2]

# Computing the Jacobian by hand
J = torch.tensor([
    [1, 0, 0, 0],
    [0, 1, 0, 0]], dtype=torch.float
)

# Confirming the above formula
y.backward(v)
(x.grad == v @ J).all()

**Remark.** Memory and compute is wasted for running code that should not involve backpropagation. Hence, we wrap this part our code in a `torch.no_grad()` context (or run it inside a function decorated with `@torch.no_grad()`) so that a computation graph is not built. 

A related method is `.detach()` used to return a tensor detached from the current graph. The result will therefore not require gradients. It is important to note that the detached tensor still shares the same storage with the original one, so that in-place modifications on either tensor takes effect for both and can result in subtle bugs.

Finally, we write our tests with `autograd` to check the correctness of our implementation:

In [27]:
x = Node(-0.4)
z = Node(0.2) * x + Node(0.1) - x
q = z.relu() ** 0.3 + z * x.tanh()
h = (z * z).relu()
y = (-h + q + q * x).tanh()
y.backward()

x_node, y_node, z_node = x, y, z
draw_graph(y_node, savename="02-test-graph")

:::{figure} ./plots/02-test-graph.svg
---
name: 02-test-graph
width: 100%
align: center
---
:::


In [28]:
x = torch.tensor(-0.4, requires_grad=True)
z = 0.2 * x + 0.1 - x
q = z.relu() ** 0.3 + z * x.tanh()
h = (z * z).relu()
y = (-h + q + q * x).tanh()

z.retain_grad()
y.retain_grad()
y.backward()

x_torch, y_torch, z_torch = x, y, z

# forward
errors = []
errors.append(abs(x_node.data - x_torch.item()))
errors.append(abs(y_node.data - y_torch.item()))
errors.append(abs(z_node.data - z_torch.item()))

# backward
errors.append(abs(x_node.grad - x_torch.grad.item()))
errors.append(abs(y_node.grad - y_torch.grad.item()))
errors.append(abs(z_node.grad - z_torch.grad.item()))

print(f"Max absolute error: {max(errors):.2e}")